In [1]:
import pandas as pd 
import numpy as np
import re
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display


In [2]:
sample_df = pd.read_csv('sample_sweetcat.csv')
sample_df.head(2)

,star,Name,gaia_dr3,Teff,eTeff,Logg,eLogg,[Fe/H],e[Fe/H],Vt,...,Gmag,Plx,Distance,Mass_t,Radius_t,SWFlag,Reference,Link,Update,Database
0,55CncA,55 Cnc,704967037090946688,5363.0,59.0,4.29,0.14,0.33,0.04,1.0,...,5.7327,79.4482,12.586818,0.922748,0.901203,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA
1,AUMic,AU Mic,6794047652729201024,3700.0,100.0,NaN,NaN,NaN,NaN,NaN,...,7.8434,102.9432,9.714095,0.512628,0.697741,0,Plavchan et al.2020,https://ui.adsabs.harvard.edu/abs/2020Natur.58...,2020-07-08,EU;NASA


In [3]:
file_path = Path('observations/Best_RM')


def read_rdb_file(file_path):
    rdb_df = pd.read_csv(file_path, sep='\t', skiprows=[1])

    for column in ['rjd', 'vrad', 'svrad']:
        rdb_df[column] = pd.to_numeric(rdb_df[column], errors='coerce')

    return rdb_df


def plot_vrad_vs_rjd(obs_name):
    obs_path = file_path / obs_name
    rdb_df = read_rdb_file(obs_path)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=rdb_df['rjd'],
            y=rdb_df['vrad'],
            error_y=dict(type='data', array=rdb_df['svrad'], visible=True),
            mode='markers+lines',
            marker=dict(size=7),
            name=obs_name,
            text=rdb_df['file_rootpath'],
            hovertemplate='rjd=%{x}<br>vrad=%{y}<br>svrad=%{error_y.array}<br>file=%{text}<extra></extra>'
        )
    )

    fig.update_layout(
        title=f'vrad vs rjd: {obs_name}',
        xaxis_title='rjd',
        yaxis_title='vrad',
        template='plotly_white',
        hovermode='closest'
    )

    fig.show()

In [4]:
all_obs_names = sorted(path.name for path in file_path.glob('*.rdb'))
obs_selector = widgets.Dropdown(options=all_obs_names, description='obs_name:')
interactive_plot = widgets.interactive_output(plot_vrad_vs_rjd, {'obs_name': obs_selector})
display(obs_selector, interactive_plot)

Dropdown(description='obs_name:', options=('HD189733_esp19_3.rdb', 'HD189733_esp19_4.rdb', 'HD209458_esp19_1.r…

Output()